In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
tmp_order = ['Start', 'Sendai', 'Skin', 'Inflamed', 'Delayed',
               'Fail1', 'Inter1', 'Inter2', 'Fail2', 'IPS']
tmp_order = tmp_order[::-1]

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome'
multiome = sc.read_h5ad(f'{indir}/h5ad/rna/raw/intersect_atac.h5ad')

In [ ]:
multiome

In [ ]:
sc.pp.normalize_total(multiome)
sc.pp.log1p(multiome)

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/csv/clusters/rna_leiden_v7.csv'

clusters = pd.read_csv(clusters_path, sep='\t', index_col=0)
clusters.shape

In [ ]:
multiome.obs['cluster'] = clusters.reindex(multiome.obs.index)['coarse_v7']

### read mct

In [ ]:
indir = '/u/project/cluo/chliu/Genomes/IGVF_hg38/annotations'
gene_annot = pd.read_csv(f'{indir}/ensembl_to_symbol.tsv', sep='\t', header=0, index_col=None)
gene_to_symbol = dict(zip(gene_annot['gene'], gene_annot['symbol']))
symbol_to_gene = dict(zip(gene_annot['symbol'], gene_annot['gene']))

In [ ]:
snmct_labels_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv'
snmct_labels = pd.read_csv(snmct_labels_path, sep='\t', index_col=0)
snmct_labels.shape

In [ ]:
snmct_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/h5ad/raw/snmct.h5ad'
snmct = sc.read_h5ad(snmct_path)

In [ ]:
snmct

In [ ]:
snmct.var['ensg'] = snmct.var.index
snmct.var['symbol'] = [gene_to_symbol.get(x) for x in snmct.var['ensg']]
snmct.var.index = snmct.var['symbol']
snmct = snmct[:,~snmct.var.index.duplicated()]

In [ ]:
snmct.var.head()

In [ ]:
snmct.obs['cluster'] = snmct_labels.reindex(snmct.obs.index)['pred_coarse']

In [ ]:
mask = ~snmct.obs['cluster'].isna()
snmct = snmct[mask]

In [ ]:
sc.pp.normalize_total(snmct)
sc.pp.log1p(snmct)

### let's look at DEGs now

In [ ]:
# ### grab top 50
# degs.shape

# top_degs.shape

In [ ]:
degs_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/csv/degs/justin_degs_v7.csv'
degs = pd.read_csv(degs_path, sep='\t', index_col=0)
degs.shape

In [ ]:
degs.head()

In [ ]:
clusts = sorted(degs['coarse_v7'].unique())
for clust in clusts:
    tmp_degs = degs[degs['coarse_v7'] == clust]
    break

In [ ]:
tmp_genes = tmp_degs.iloc[:50].index
tmp_mult_genes = tmp_genes[tmp_genes.isin(multiome.var.index)]
tmp_mult_genes.shape

In [ ]:
tmp_mct_genes = tmp_genes[tmp_genes.isin(snmct.var.index)]
tmp_mct_genes.shape

In [ ]:
tmp_mct_genes = tmp_mct_genes[tmp_mct_genes.isin(tmp_mult_genes)]
tmp_mult_genes = tmp_mult_genes[tmp_mult_genes.isin(tmp_mct_genes)]
tmp_mct_genes.shape, tmp_mult_genes.shape

In [ ]:
cat_order = multiome.obs['cluster'].value_counts().sort_index().index

In [ ]:
sc.pl.dotplot(multiome, var_names = tmp_mult_genes, groupby='cluster',
              categories_order = cat_order)

In [ ]:
sc.pl.dotplot(snmct, var_names = tmp_mult_genes, groupby='cluster',
              categories_order = cat_order)

### let's look at each cluster

In [ ]:
multiome.obs['cluster'] = pd.Categorical(multiome.obs['cluster'], categories=tmp_order, ordered=True)
snmct.obs['cluster'] = pd.Categorical(snmct.obs['cluster'], categories=tmp_order, ordered=True)

In [ ]:
clusts = sorted(degs['coarse_v7'].unique())
n_genes = 100

for clust in clusts:
    print(clust)
    tmp_degs = degs[degs['coarse_v7'] == clust]
    
    ###
    tmp_genes = tmp_degs.iloc[:n_genes].index
    tmp_mult_genes = tmp_genes[tmp_genes.isin(multiome.var.index)]
    tmp_mct_genes = tmp_genes[tmp_genes.isin(snmct.var.index)]
    tmp_mct_genes = tmp_mct_genes[tmp_mct_genes.isin(tmp_mult_genes)]
    tmp_mult_genes = tmp_mult_genes[tmp_mult_genes.isin(tmp_mct_genes)]
    
    ###
    sc.pl.dotplot(multiome, var_names = tmp_mult_genes, groupby='cluster',
                  categories_order = cat_order)

    sc.pl.dotplot(snmct, var_names = tmp_mult_genes, groupby='cluster',
                  categories_order = cat_order)

In [ ]:
# ### ok pick like 5 genes lmao

In [ ]:
#                      inter1_genes, inter2_genes, fail1_genes, fail2_genes, ips_genes])

In [ ]:
select_degs = ['CEBPD', 'CEMIP', # Start
              'KLF4', 'IL31RA',# Sendai
              'KLK14', 'IGF2', # Skin
              'PMAIP1', 'COL4A3', 'HIP1R', # Inflamed
#               'HMGA1', 'IL6', # Delayed
               'IFI6', 'COL5A3', # Fail1
#                'NFIA', 'TCF7L2', # Inter1
#                'XACT', 'LEFTY2', 'TENM1', # Inter2
                'LEFTY2', # Inter2
#                'MEOX2','SNTG1', # Fail2
               'MEOX2', # Fail2
               'DNMT3B', 'EPCAM', 'CTSV', # IPS
              ]

select_degs = pd.Index(select_degs)
mask = select_degs.isin(snmct.var.index)
final_genes = select_degs[mask]
final_genes.shape

In [ ]:
sc.pl.dotplot(multiome, var_names = final_genes, groupby='cluster',
              categories_order = cat_order, title='', vmax=1.5)

sc.pl.dotplot(snmct, var_names = final_genes, groupby='cluster',
              categories_order = cat_order, title='', vmax=1.5)

### what about old genes?

In [ ]:
select_degs = ['CEMIP', 'PTGIS', 'EFEMP1', 'IGFGP3',
               'KLF4', 'ELN',
               'KLK11', 'SPPR3',
               'EGF', 'KCNQ4', 'COL4A3',
               'IFI6', 'COL5A3', 'PDGFRB',
               'GABRA2', 'GRM1',
               'ACOXL', 'LINC01950',
               'STNG1', 'NTRK2'       
              ]

select_degs = pd.Index(select_degs)
mask = select_degs.isin(snmct.var.index)
final_genes = select_degs[mask]
final_genes.shape

In [ ]:
sc.pl.dotplot(multiome, var_names = final_genes, groupby='cluster',
              categories_order = cat_order, title='', vmax=1.5)

sc.pl.dotplot(snmct, var_names = final_genes, groupby='cluster',
              categories_order = cat_order, title='', vmax=1.5)

### write out

In [ ]:
select_degs = ['CEBPD', 'CEMIP', # Start
              'KLF4', 'IL31RA',# Sendai
              'KLK14', 'IGF2', # Skin
              'PMAIP1', 'COL4A3', 'HIP1R', # Inflamed
#               'HMGA1', 'IL6', # Delayed
               'IFI6', 'COL5A3', # Fail1
#                'NFIA', 'TCF7L2', # Inter1
#                'XACT', 'LEFTY2', 'TENM1', # Inter2
                'LEFTY2', # Inter2
#                'MEOX2','SNTG1', # Fail2
               'MEOX2', # Fail2
               'DNMT3B', 'EPCAM', 'CTSV', # IPS
              ]

select_degs = pd.Index(select_degs)
mask = select_degs.isin(snmct.var.index)
final_genes = select_degs[mask]
final_genes.shape

In [ ]:
final_genes

In [ ]:
mask = multiome.var.index.isin(final_genes)
tmp_multiome = multiome[:,mask]
multiome_df = pd.DataFrame(tmp_multiome.X.toarray())
multiome_df.index = tmp_multiome.obs.index
multiome_df.columns = tmp_multiome.var.index
multiome_df = multiome_df.loc[:,final_genes]

multiome_meta = multiome.obs[['cluster']]
multiome_df.shape, multiome_meta.shape

In [ ]:
mask = snmct.var.index.isin(final_genes)
tmp_snmct = snmct[:,mask]
snmct_df = pd.DataFrame(tmp_snmct.X.toarray())
snmct_df.index = tmp_snmct.obs.index
snmct_df.columns = tmp_snmct.var.index
snmct_df = snmct_df.loc[:,final_genes]

snmct_meta = snmct.obs[['cluster']]
snmct_df.shape, snmct_meta.shape

In [ ]:
multiome_df.head()

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure1'
multiome_df.to_csv(f'{indir}/multiome_compare_dot_plot.csv', sep='\t')
multiome_meta.to_csv(f'{indir}/multiome_compare_dot_plot_meta.csv', sep='\t')

snmct_df.to_csv(f'{indir}/snmct_compare_dot_plot.csv', sep='\t')
snmct_meta.to_csv(f'{indir}/snmct_compare_dot_plot_meta.csv', sep='\t')

### weird

In [ ]:
sc.pl.dotplot(multiome, var_names = final_genes, groupby='cluster',
              categories_order = cat_order, title='', vmax=2)

sc.pl.dotplot(snmct, var_names = final_genes, groupby='cluster',
              categories_order = cat_order, title='', vmax=2)

In [ ]:
!date